# Nomadic Intelligence — §4.9 Task Generalization Experiment

**목적**: §4.1 결과가 선형/gradual/Gaussian 태스크 설계에 종속되지 않음을 검증.

| Variant | 변경 내용 | 검증 주장 |
|---|---|---|
| `nonlinear` | regime 함수 → 비선형 (sin/quadratic/tanh) | 선형 분리 가능성 의존 아님 |
| `abrupt` | transition_steps = 2 | 급격한 전환에서도 switch latency 유지 |
| `gradual` | transition_steps = 24 | 느린 전환에서도 fixation 패턴 유지 |
| `heavy_tail` | noise → Student-T (df=2) | 극단값 노이즈에서도 Δx 신호 유효 |
| `combined` | 비선형 + abrupt(steps=2) + heavy-tail | 모든 조건 동시 적용 |

**비교 모델**: Fixed / Standard MoE / Nomadic Full (3-seed: 42, 123, 456)

**기준값 (§4.1 원본)**:

| Model | Seq MSE | ΔH |
|---|---|---|
| Fixed | 0.409 | — |
| Standard MoE | 0.410 | 0.017 |
| Nomadic Full | 0.165 | 0.781 |

**GPU**: L4


In [ ]:
# STEP 0: 환경 확인
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# ============================================================
# STEP 1: 실험 조건 설정 — 이 셀만 바꿔서 5개 variant 전환
# ============================================================
# 선택지: 'nonlinear' | 'abrupt' | 'gradual' | 'heavy_tail' | 'combined'
TASK_VARIANT = 'nonlinear'

VARIANT_DESC = {
    'nonlinear':   'Nonlinear regime functions (sin/quadratic/tanh), Gaussian noise, steps=8',
    'abrupt':      'Linear functions, Gaussian noise, transition_steps=2 (abrupt)',
    'gradual':     'Linear functions, Gaussian noise, transition_steps=24 (gradual)',
    'heavy_tail':  'Linear functions, Student-T noise (df=2), steps=8',
    'combined':    'Nonlinear + abrupt(steps=2) + heavy-tail noise',
}

# 기준값 (§4.1 원본, 비교용)
BASELINE_RESULTS = {
    'Fixed':       {'seq_mse': 0.409, 'dh': None},
    'StdMoE':      {'seq_mse': 0.410, 'dh': 0.017},
    'NomadicFull': {'seq_mse': 0.165, 'dh': 0.781},
}

print(f'Variant: {TASK_VARIANT}')
print(f'Description: {VARIANT_DESC[TASK_VARIANT]}')


In [ ]:
# STEP 2: Imports & Reproducibility
import os, random, math
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.rcParams['figure.dpi'] = 120

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS  = [42, 123, 456]
print(f'Device: {DEVICE}')


In [ ]:
# STEP 3: Config (config.yaml 기준값 유지, beta_phi=0.02)
@dataclass
class Config:
    seed:   int   = 42
    device: str   = DEVICE

    input_dim:  int   = 2
    output_dim: int   = 1
    overlap_std: float = 0.9       # Gaussian 기준; heavy_tail은 noise_fn으로 override

    hidden_dim:      int   = 64
    num_experts:     int   = 3
    gate_hidden_dim: int   = 64
    temperature:     float = 0.60

    epochs:       int   = 220
    lr:           float = 2e-3
    weight_decay: float = 1e-5

    phase_batch_size:    int = 64
    phase_train_cycles:  int = 40
    phase_test_cycles:   int = 12
    transition_steps:    int = 8       # abrupt=2 / gradual=24 / combined=2

    ema_decay:              float = 0.80
    err_baseline_momentum:  float = 0.85
    w_env: float = 1.0
    w_err: float = 2.0

    alpha_dogma:     float = 0.04
    beta_nomad:      float = 0.05
    beta_phi:        float = 0.02
    gamma_diversity: float = 0.08
    lambda_sep:      float = 0.08
    lambda_cons:     float = 0.03
    lambda_load:     float = 0.03
    tau_k_min:       int   = 3
    tau_k_penalty:   float = 0.05

    use_dynamic_tau: bool  = True
    tau_min:   float = 2.0
    tau_max:   float = 8.0
    tau_var_scale:  float = 6.0
    tau_var_window: int   = 8

    phi_scale_env:     float = 1.0
    phi_scale_err:     float = 1.5
    phi_scale_explain: float = 1.5
    phi_scale_gap:     float = 0.8

    temp_stable:     float = 0.35
    temp_transition: float = 0.90

    use_hard_switch:     bool  = True
    phi_hard_threshold:  float = 0.30

    policy_hidden_dim:       int   = 64
    policy_mix_weight:       float = 0.25
    policy_weight_stay:      float = 0.20
    policy_weight_target:    float = 0.20
    policy_weight_mode:      float = 0.10
    policy_switch_threshold: float = 0.50


def make_config(seed: int, variant: str) -> Config:
    cfg = Config(seed=seed)
    if variant == 'abrupt':
        cfg.transition_steps = 2
    elif variant == 'gradual':
        cfg.transition_steps = 24
    elif variant == 'combined':
        cfg.transition_steps = 2
    return cfg

print('Config ready.')


In [ ]:
# STEP 4: Data Generation — variant별 분기
REGIME_TO_ID = {'A': 0, 'B': 1, 'C': 2}
ID_TO_REGIME = {0: 'A', 1: 'B', 2: 'C'}
REGIME_ORDER = ['A', 'B', 'C']

# ── 노이즈 샘플러
def sample_noise(n: int, std: float, device: str, variant: str) -> torch.Tensor:
    """Gaussian (default) or Student-T df=2 (heavy_tail / combined)."""
    if variant in ('heavy_tail', 'combined'):
        # Student-T df=2: 분산이 정의되나 꼬리가 두꺼움
        dist = torch.distributions.StudentT(df=2.0)
        noise = dist.sample((n, 2)).to(device) * std
    else:
        noise = std * torch.randn(n, 2, device=device)
    return noise

def sample_regime_x(regime: str, n: int, cfg: Config, variant: str) -> torch.Tensor:
    centers = {'A': (2.5, 2.5), 'B': (-2.5, -2.5), 'C': (2.5, -2.5)}
    c = torch.tensor(centers[regime], device=cfg.device)
    noise = sample_noise(n, cfg.overlap_std, cfg.device, variant)
    return noise + c

# ── regime 함수
def regime_function(x: torch.Tensor, regime: str, variant: str) -> torch.Tensor:
    x1, x2 = x[:, 0], x[:, 1]
    if variant in ('nonlinear', 'combined'):
        # Option A: 중간 난이도 비선형
        if regime == 'A':   y = torch.sin(x1) * x2
        elif regime == 'B': y = x1 ** 2 - x2 ** 2
        elif regime == 'C': y = torch.tanh(x1 + x2) * x2.abs()
    else:
        # 원본 선형 함수 (§4.1 동일)
        if regime == 'A':   y = x1 + x2
        elif regime == 'B': y = x1 - x2
        elif regime == 'C': y = -x1 + 0.5 * x2
    return y.unsqueeze(-1)

def generate_phase_sequence(cfg: Config, cycles: int, variant: str):
    xs, ys, rs, tags = [], [], [], []
    for _ in range(cycles):
        for i, curr_r in enumerate(REGIME_ORDER):
            next_r = REGIME_ORDER[(i + 1) % 3]
            # stable block
            x_s = sample_regime_x(curr_r, cfg.phase_batch_size, cfg, variant)
            y_s = regime_function(x_s, curr_r, variant)
            r_s = torch.full((cfg.phase_batch_size,), REGIME_TO_ID[curr_r],
                             dtype=torch.long, device=cfg.device)
            xs.append(x_s); ys.append(y_s); rs.append(r_s)
            tags.extend([f'stable_{curr_r}'] * cfg.phase_batch_size)
            # transition block
            for step in range(cfg.transition_steps):
                alpha = (step + 1) / cfg.transition_steps
                x_a = sample_regime_x(curr_r, cfg.phase_batch_size, cfg, variant)
                x_b = sample_regime_x(next_r, cfg.phase_batch_size, cfg, variant)
                x_mix = (1 - alpha) * x_a + alpha * x_b
                y_mix = ((1 - alpha) * regime_function(x_mix, curr_r, variant)
                         + alpha    * regime_function(x_mix, next_r, variant))
                dominant = curr_r if alpha < 0.5 else next_r
                r_mix = torch.full((cfg.phase_batch_size,), REGIME_TO_ID[dominant],
                                   dtype=torch.long, device=cfg.device)
                xs.append(x_mix); ys.append(y_mix); rs.append(r_mix)
                tags.extend([f'transition_{curr_r}_to_{next_r}'] * cfg.phase_batch_size)
    return torch.cat(xs), torch.cat(ys), torch.cat(rs), tags

def iterate_sequence_minibatches(X, Y, R, batch_size):
    n = X.size(0)
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        yield X[s:e], Y[s:e], R[s:e]

print('Data utilities ready.')
print(f'  Regime function: {"nonlinear" if TASK_VARIANT in ("nonlinear","combined") else "linear"}')
print(f'  Noise:           {"Student-T df=2" if TASK_VARIANT in ("heavy_tail","combined") else "Gaussian"}')
print(f'  transition_steps: abrupt=2 / gradual=24 / others=8')


In [ ]:
# STEP 5: Model Definitions (run_structured.py 동일)
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x): return self.net(x)

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x): return self.net(x)

class StandardMoE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts, gate_hidden_dim):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim, output_dim)
                                      for _ in range(num_experts)])
        self.gate = nn.Sequential(
            nn.Linear(input_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, num_experts),
        )
    def forward(self, x, hard=False):
        gate_probs = F.softmax(self.gate(x), dim=-1)
        expert_outputs = torch.stack([e(x) for e in self.experts], dim=1)
        routing = (F.one_hot(gate_probs.argmax(-1), self.num_experts).float()
                   if hard else gate_probs)
        return (routing.unsqueeze(-1) * expert_outputs).sum(1), gate_probs, None, expert_outputs

class NomadicMoE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts,
                 gate_hidden_dim, policy_hidden_dim=64):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim, output_dim)
                                      for _ in range(num_experts)])
        self.gate = nn.Sequential(
            nn.Linear(input_dim + 2, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, num_experts),
        )
        policy_in = input_dim + 5
        self.policy_shared = nn.Sequential(
            nn.Linear(policy_in, policy_hidden_dim), nn.ReLU(),
            nn.Linear(policy_hidden_dim, policy_hidden_dim), nn.ReLU(),
        )
        self.stay_head   = nn.Linear(policy_hidden_dim, 2)
        self.target_head = nn.Linear(policy_hidden_dim, num_experts)
        self.mode_head   = nn.Linear(policy_hidden_dim, 2)

    def gate_forward(self, x, dh, de, temperature):
        gate_input = torch.cat([x, dh, de], dim=-1)
        gate_probs = F.softmax(self.gate(gate_input) / temperature, dim=-1)
        return gate_probs

    def policy_forward(self, policy_input):
        h = self.policy_shared(policy_input)
        return (F.softmax(self.stay_head(h),   dim=-1),
                F.softmax(self.target_head(h), dim=-1),
                F.softmax(self.mode_head(h),   dim=-1))

    def forward(self, x, delta_hybrid, delta_err, temperature, hard=False):
        gate_probs = self.gate_forward(x, delta_hybrid, delta_err, temperature)
        expert_outputs = torch.stack([e(x) for e in self.experts], dim=1)
        routing = (F.one_hot(gate_probs.argmax(-1), self.num_experts).float()
                   if hard else gate_probs)
        return (routing.unsqueeze(-1) * expert_outputs).sum(1), gate_probs, None, expert_outputs

print('Models ready.')


In [ ]:
# STEP 6: Utilities (run_structured.py 동일)
class HybridDeltaTracker:
    def __init__(self, cfg):
        self.cfg = cfg
        self.prev_x_mean = None; self.err_ema = None; self.err_baseline = None
        self.recent_delta_env = deque(maxlen=cfg.tau_var_window)

    def reset(self):
        self.prev_x_mean = None; self.err_ema = None; self.err_baseline = None
        self.recent_delta_env.clear()

    def compute_dynamic_tau(self, sigma2):
        tau = self.cfg.tau_min + (self.cfg.tau_max - self.cfg.tau_min) / (
              1.0 + self.cfg.tau_var_scale * sigma2)
        return float(np.clip(tau, self.cfg.tau_min, self.cfg.tau_max))

    def compute(self, x, batch_mse):
        x_mean = x.mean(0, keepdim=True)
        de = 0.0 if self.prev_x_mean is None else float(
             torch.norm(x_mean - self.prev_x_mean, p=2).item())
        be = batch_mse.detach()
        if self.err_ema is None:
            self.err_ema = be; self.err_baseline = be; derr = 0.0
        else:
            self.err_ema = self.cfg.ema_decay * self.err_ema + (1 - self.cfg.ema_decay) * be
            self.err_baseline = (self.cfg.err_baseline_momentum * self.err_baseline
                                 + (1 - self.cfg.err_baseline_momentum) * self.err_ema)
            derr = float(torch.relu(self.err_ema - self.err_baseline).item())
        dh = float(torch.tanh(torch.tensor(
             self.cfg.w_env * de + self.cfg.w_err * derr)).item())
        self.prev_x_mean = x_mean.detach()
        self.recent_delta_env.append(de)
        sigma2 = float(np.var(self.recent_delta_env)) if len(self.recent_delta_env) >= 2 else 0.0
        dyn_tau = self.compute_dynamic_tau(sigma2)
        delta_hybrid = torch.full((x.size(0), 1), dh, device=self.cfg.device)
        return delta_hybrid, de, derr, dh, sigma2, dyn_tau


class DwellTimeRegularizer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.current_expert = None; self.dwell_count = 0

    def reset(self):
        self.current_expert = None; self.dwell_count = 0

    def compute(self, gate_probs, tau_dynamic=None):
        dominant = int(torch.bincount(gate_probs.argmax(-1),
                        minlength=gate_probs.size(-1)).argmax().item())
        if dominant == self.current_expert: self.dwell_count += 1
        else: self.current_expert = dominant; self.dwell_count = 1
        eps = 1e-8
        entropy = -(gate_probs * (gate_probs + eps).log()).sum(-1).mean()
        tau_cap = float(tau_dynamic if tau_dynamic is not None else self.cfg.tau_k_min)
        if self.dwell_count <= tau_cap:
            return -self.cfg.tau_k_penalty * entropy
        else:
            excess = self.dwell_count - tau_cap
            return min(float(excess) * self.cfg.tau_k_penalty,
                       self.cfg.tau_k_penalty * 10) * entropy


def gate_entropy(gate_probs):
    eps = 1e-8
    return -(gate_probs * (gate_probs + eps).log()).sum(-1)

def compute_load_balancing_loss(gate_probs):
    K = gate_probs.size(-1)
    mean_gate = gate_probs.mean(0)
    top1 = gate_probs.argmax(-1)
    top1_frac = torch.bincount(top1, minlength=K).float() / top1.size(0)
    return K * (top1_frac * mean_gate).sum()

def compute_dogma_penalty(gate_probs):
    mu = gate_probs.mean(0)
    return mu.pow(2).sum() - 1.0 / gate_probs.size(1)

def compute_nomad_bonus(gate_probs):
    eps = 1e-8
    return -(gate_probs * (gate_probs + eps).log()).sum(-1).mean()

def compute_diversity_loss(expert_outputs):
    K = expert_outputs.size(1)
    if K < 2: return expert_outputs.new_zeros(1).squeeze()
    idx_i, idx_j = zip(*[(i,j) for i in range(K) for j in range(i+1,K)])
    return F.cosine_similarity(expert_outputs[:, idx_i, :],
                               expert_outputs[:, idx_j, :], dim=-1).mean()

def compute_explanation_signals(y_true, y_hat, expert_outputs, gate_probs):
    expl_err = F.mse_loss(y_hat, y_true)
    per_err  = ((expert_outputs - y_true.unsqueeze(1)) ** 2).mean(-1)
    top1_err = per_err.gather(1, gate_probs.argmax(-1).unsqueeze(1)).mean()
    best_err = per_err.min(1).values.mean()
    return expl_err, torch.relu(top1_err - best_err)

def compute_phi(de, derr, expl_err, gap, cfg):
    dev = expl_err.device
    return torch.tanh(
        cfg.phi_scale_env     * torch.tensor(de,   device=dev)
      + cfg.phi_scale_err     * torch.tensor(derr, device=dev)
      + cfg.phi_scale_explain * expl_err.detach()
      + cfg.phi_scale_gap     * gap.detach())

def compute_temp(phi, cfg):
    return cfg.temp_stable + (cfg.temp_transition - cfg.temp_stable) * float(phi.mean().item())

def build_policy_input(xb, dh_t, de_t, phi, sigma2, dyn_tau):
    x_sum = xb.mean(0, keepdim=True).expand(xb.size(0), -1)
    phi_t = torch.full((xb.size(0),1), float(phi.mean().item()), device=xb.device)
    s2_t  = torch.full((xb.size(0),1), float(np.tanh(sigma2*10.0)), device=xb.device)
    tau_t = torch.full((xb.size(0),1), float(np.tanh((dyn_tau-5.0)/5.0)), device=xb.device)
    return torch.cat([x_sum, dh_t, de_t, phi_t, s2_t, tau_t], dim=-1)

def build_policy_targets(yb, exp_out, phi, sigma2, dyn_tau, cfg):
    per_err = ((exp_out - yb.unsqueeze(1))**2).mean(-1)
    tgt = per_err.mean(0).argmin().long()
    phi_val = float(phi.mean().item())
    sw  = 1 if (phi_val > cfg.policy_switch_threshold or sigma2 > 0.05) else 0
    mod = 1 if (phi_val <= cfg.policy_switch_threshold and dyn_tau >= 5.5) else 0
    return sw, tgt, mod

def compute_regime_gate_stats(gate_probs, regime_ids):
    dev = gate_probs.device
    valid_means = []
    l_cons = torch.tensor(0.0, device=dev); cnt = 0
    for rid in range(3):
        mask = regime_ids == rid
        if mask.sum() == 0: continue
        g_r = gate_probs[mask]; u_r = g_r.mean(0)
        valid_means.append(u_r)
        l_cons = l_cons + ((g_r - u_r.unsqueeze(0))**2).sum(-1).mean(); cnt += 1
    if cnt > 0: l_cons = l_cons / cnt
    if len(valid_means) < 2:
        return torch.tensor(0.0, device=dev), l_cons
    pairwise = [torch.norm(valid_means[i]-valid_means[j], p=2)
                for i in range(len(valid_means)) for j in range(i+1, len(valid_means))]
    return -torch.stack(pairwise).mean(), l_cons

def regimewise_usage(gate_probs, regime_ids, num_experts):
    top1 = gate_probs.argmax(-1); usage = {}
    for rid in range(3):
        mask = regime_ids == rid; name = ID_TO_REGIME[rid]
        if mask.sum() == 0: usage[name] = np.zeros(num_experts); continue
        c = torch.bincount(top1[mask], minlength=num_experts).float()
        usage[name] = (c / c.sum().clamp_min(1.0)).cpu().numpy()
    return usage

def infer_regime_to_expert(usage):
    return {r: int(np.argmax(usage[r])) for r in ['A','B','C']}

def compute_switch_latency(regime_seq, top1_seq, r2e):
    lats = []; prev = regime_seq[0] if regime_seq else None
    for t in range(1, len(regime_seq)):
        curr = regime_seq[t]
        if curr != prev:
            tgt = r2e.get(curr)
            if tgt is not None:
                for k in range(t, len(top1_seq)):
                    if int(top1_seq[k]) == int(tgt): lats.append(k-t); break
        prev = curr
    return lats

def compute_dwell_times(seq):
    if len(seq) == 0: return []
    dwells=[]; cur=seq[0]; run=1
    for t in range(1, len(seq)):
        if seq[t]==cur: run+=1
        else: dwells.append(run); cur=seq[t]; run=1
    dwells.append(run); return dwells

print('Utilities ready.')


In [ ]:
# STEP 7: Evaluation Functions
def eval_fixed_seq(model, X, Y, R, cfg):
    model.eval()
    with torch.no_grad():
        return F.mse_loss(model(X), Y).item()

def eval_stdmoe_seq(model, X, Y, R, phase_tags, cfg):
    model.eval()
    all_y, all_g, tags, ents = [], [], [], []
    with torch.no_grad():
        for bi, (xb,yb,rb) in enumerate(iterate_sequence_minibatches(X,Y,R,cfg.phase_batch_size)):
            y_hat, gp, _, _ = model(xb)
            all_y.append(y_hat); all_g.append(gp)
            tags.append(phase_tags[bi*cfg.phase_batch_size])
            ents.append(gate_entropy(gp).mean().item())
    seq_mse = F.mse_loss(torch.cat(all_y), Y).item()
    sh = [e for t,e in zip(tags,ents) if t.startswith('stable_')]
    th = [e for t,e in zip(tags,ents) if t.startswith('transition_')]
    return seq_mse, {
        'stable_entropy_mean':     float(np.mean(sh)) if sh else float('nan'),
        'transition_entropy_mean': float(np.mean(th)) if th else float('nan'),
        'delta_h': float(np.mean(th)-np.mean(sh)) if (sh and th) else float('nan'),
    }

def eval_nomadic_seq(model, X, Y, R, phase_tags, cfg):
    model.eval()
    tracker = HybridDeltaTracker(cfg); tracker.reset()
    all_y, all_g, tags, ents, top1_list, reg_list = [], [], [], [], [], []
    with torch.no_grad():
        for bi, (xb,yb,rb) in enumerate(iterate_sequence_minibatches(X,Y,R,cfg.phase_batch_size)):
            z = torch.zeros((xb.size(0),1), device=cfg.device)
            warm_mse = F.mse_loss(model(xb,z,z,cfg.temperature)[0], yb)
            dh_t, de, derr, dh, sigma2, dyn_tau = tracker.compute(xb, warm_mse)
            de_t = torch.full((xb.size(0),1), derr, device=cfg.device)

            p_y, p_g, _, p_e = model(xb, dh_t, de_t, cfg.temperature)
            expl, gap = compute_explanation_signals(yb, p_y, p_e, p_g)
            phi = compute_phi(de, derr, expl, gap, cfg)
            temp = compute_temp(phi, cfg)

            y_hat, gp, _, exp_out = model(xb, dh_t, de_t, temp)

            # PolicyNet
            pol_in = build_policy_input(xb, dh_t, de_t, phi, sigma2, dyn_tau)
            sw_p, tgt_p, mode_p = model.policy_forward(pol_in)
            eff_mix = cfg.policy_mix_weight * float(sw_p[:,1].mean().item())
            tgt_idx = tgt_p.mean(0).argmax()
            tgt_oh  = F.one_hot(tgt_idx, cfg.num_experts).float().unsqueeze(0).expand(xb.size(0),-1)
            tgt_ste = (tgt_oh - gp).detach() + gp
            mixed   = (1-eff_mix)*gp + eff_mix*tgt_ste
            failsafe = dh > cfg.phi_hard_threshold
            hard_mode = cfg.use_hard_switch and (mode_p[:,1].mean().item()>0.5) and not failsafe
            final_r = F.one_hot(mixed.argmax(-1), cfg.num_experts).float() if hard_mode else mixed
            y_hat = (final_r.unsqueeze(-1)*exp_out).sum(1); gp = final_r

            all_y.append(y_hat); all_g.append(gp)
            tags.append(phase_tags[bi*cfg.phase_batch_size])
            ents.append(gate_entropy(gp).mean().item())
            top1_list.append(int(torch.bincount(gp.argmax(-1), minlength=cfg.num_experts).argmax().item()))
            reg_list.append(ID_TO_REGIME[int(rb[0].item())])

    seq_mse = F.mse_loss(torch.cat(all_y), Y).item()
    G = torch.cat(all_g)
    usage = regimewise_usage(G, R, cfg.num_experts)
    r2e = infer_regime_to_expert(usage)
    lats = compute_switch_latency(reg_list, np.array(top1_list), r2e)
    sh = [e for t,e in zip(tags,ents) if t.startswith('stable_')]
    th = [e for t,e in zip(tags,ents) if t.startswith('transition_')]
    return seq_mse, {
        'stable_entropy_mean':     float(np.mean(sh)) if sh else float('nan'),
        'transition_entropy_mean': float(np.mean(th)) if th else float('nan'),
        'delta_h': float(np.mean(th)-np.mean(sh)) if (sh and th) else float('nan'),
        'mean_switch_latency': float(np.mean(lats)) if lats else float('nan'),
    }

print('Evaluation functions ready.')


In [ ]:
# STEP 8: Training Functions
def train_fixed(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te):
    model = MLPRegressor(cfg.input_dim, cfg.hidden_dim, cfg.output_dim).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse_log = []
    for ep in range(cfg.epochs):
        model.train()
        for xb,yb,_ in iterate_sequence_minibatches(Xtr,Ytr,Rtr,cfg.phase_batch_size):
            opt.zero_grad(); F.mse_loss(model(xb),yb).backward(); opt.step()
        mse_log.append(eval_fixed_seq(model, Xte, Yte, Rte, cfg))
        if (ep+1) % 55 == 0 or ep == 0:
            print(f'  [Fixed] Ep {ep+1:03d} | MSE {mse_log[-1]:.4f}')
    return model, mse_log

def train_stdmoe(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te):
    model = StandardMoE(cfg.input_dim, cfg.hidden_dim, cfg.output_dim,
                        cfg.num_experts, cfg.gate_hidden_dim).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse_log, dyn_log = [], []
    for ep in range(cfg.epochs):
        model.train()
        for xb,yb,_ in iterate_sequence_minibatches(Xtr,Ytr,Rtr,cfg.phase_batch_size):
            opt.zero_grad()
            y_hat, gp, _, exp = model(xb)
            loss = (F.mse_loss(y_hat,yb)
                    + cfg.gamma_diversity * compute_diversity_loss(exp)
                    + cfg.lambda_load     * compute_load_balancing_loss(gp))
            loss.backward(); opt.step()
        seq_mse, dyn = eval_stdmoe_seq(model, Xte, Yte, Rte, tags_te, cfg)
        mse_log.append(seq_mse); dyn_log.append(dyn)
        if (ep+1) % 55 == 0 or ep == 0:
            print(f'  [StdMoE] Ep {ep+1:03d} | MSE {seq_mse:.4f} '
                  f'| ΔH {dyn["delta_h"]:.3f}')
    return model, mse_log, dyn_log

def train_nomadic(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te):
    model = NomadicMoE(cfg.input_dim, cfg.hidden_dim, cfg.output_dim,
                       cfg.num_experts, cfg.gate_hidden_dim,
                       cfg.policy_hidden_dim).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse_log, dyn_log = [], []

    for ep in range(cfg.epochs):
        model.train()
        tracker  = HybridDeltaTracker(cfg); tracker.reset()
        dwell_reg = DwellTimeRegularizer(cfg); dwell_reg.reset()

        for xb,yb,rb in iterate_sequence_minibatches(Xtr,Ytr,Rtr,cfg.phase_batch_size):
            opt.zero_grad()
            z = torch.zeros((xb.size(0),1), device=cfg.device)
            with torch.no_grad():
                warm_mse = F.mse_loss(model(xb,z,z,cfg.temperature)[0], yb)
            dh_t, de, derr, dh, sigma2, dyn_tau = tracker.compute(xb, warm_mse)
            de_t = torch.full((xb.size(0),1), derr, device=cfg.device)

            with torch.no_grad():
                p_y, p_g, _, p_e = model(xb, dh_t, de_t, cfg.temperature)
            expl, gap = compute_explanation_signals(yb, p_y, p_e, p_g)
            phi  = compute_phi(de, derr, expl, gap, cfg)
            temp = compute_temp(phi, cfg)

            pol_in = build_policy_input(xb, dh_t, de_t, phi, sigma2, dyn_tau)
            sw_p, tgt_p, mode_p = model.policy_forward(pol_in)

            y_hat, gp, _, exp_out = model(xb, dh_t, de_t, temp)
            eff_mix = cfg.policy_mix_weight * float(sw_p[:,1].mean().item())
            tgt_idx = tgt_p.mean(0).argmax()
            tgt_oh  = F.one_hot(tgt_idx, cfg.num_experts).float().unsqueeze(0).expand(xb.size(0),-1)
            tgt_ste = (tgt_oh - gp).detach() + gp
            mixed   = (1-eff_mix)*gp + eff_mix*tgt_ste
            failsafe = dh > cfg.phi_hard_threshold
            hard_mode = cfg.use_hard_switch and (mode_p[:,1].mean().item()>0.5) and not failsafe
            final_r = F.one_hot(mixed.argmax(-1), cfg.num_experts).float() if hard_mode else mixed
            y_hat = (final_r.unsqueeze(-1)*exp_out).sum(1)

            _, gap2 = compute_explanation_signals(yb, y_hat, exp_out, final_r)
            sep_l, cons_l = compute_regime_gate_stats(final_r, rb)
            tau_d = dyn_tau if cfg.use_dynamic_tau else float(cfg.tau_k_min)
            dw    = dwell_reg.compute(final_r, tau_dynamic=tau_d)

            sw_t  = torch.full((xb.size(0),), build_policy_targets(yb,p_e,phi,sigma2,dyn_tau,cfg)[0], dtype=torch.long, device=cfg.device)
            tgt_t = torch.full((xb.size(0),), int(build_policy_targets(yb,p_e,phi,sigma2,dyn_tau,cfg)[1].item()), dtype=torch.long, device=cfg.device)
            mod_t = torch.full((xb.size(0),), build_policy_targets(yb,p_e,phi,sigma2,dyn_tau,cfg)[2], dtype=torch.long, device=cfg.device)

            loss = (F.mse_loss(y_hat, yb)
                  + cfg.beta_phi        * (phi.detach() * gap2)
                  + cfg.alpha_dogma     * compute_dogma_penalty(final_r)
                  - cfg.beta_nomad      * compute_nomad_bonus(final_r)
                  + cfg.gamma_diversity * compute_diversity_loss(exp_out)
                  + cfg.lambda_sep      * sep_l
                  + cfg.lambda_cons     * cons_l
                  + cfg.lambda_load     * compute_load_balancing_loss(final_r)
                  + cfg.policy_weight_stay   * F.nll_loss(torch.log(sw_p   +1e-8), sw_t)
                  + cfg.policy_weight_target * F.nll_loss(torch.log(tgt_p  +1e-8), tgt_t)
                  + cfg.policy_weight_mode   * F.nll_loss(torch.log(mode_p +1e-8), mod_t)
                  - dw)
            loss.backward(); opt.step()

        seq_mse, dyn = eval_nomadic_seq(model, Xte, Yte, Rte, tags_te, cfg)
        mse_log.append(seq_mse); dyn_log.append(dyn)
        if (ep+1) % 55 == 0 or ep == 0:
            print(f'  [Nomadic] Ep {ep+1:03d} | MSE {seq_mse:.4f} '
                  f'| StH {dyn["stable_entropy_mean"]:.3f} '
                  f'| TrH {dyn["transition_entropy_mean"]:.3f} '
                  f'| ΔH {dyn["delta_h"]:.3f}')
    return model, mse_log, dyn_log

print('Training functions ready.')


In [ ]:
# STEP 9: 실험 실행 — 3 models x 3 seeds
import time

all_results = {'Fixed': {}, 'StdMoE': {}, 'Nomadic': {}}

for seed in SEEDS:
    t0 = time.time()
    print(f'\n========== Seed {seed} | variant={TASK_VARIANT} ==========')
    set_seed(seed)
    cfg = make_config(seed, TASK_VARIANT)

    Xtr, Ytr, Rtr, tags_tr = generate_phase_sequence(cfg, cfg.phase_train_cycles, TASK_VARIANT)
    Xte, Yte, Rte, tags_te = generate_phase_sequence(cfg, cfg.phase_test_cycles,  TASK_VARIANT)

    print('--- Fixed ---')
    _, mse_log = train_fixed(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te)
    all_results['Fixed'][seed] = {'mse_log': mse_log, 'dyn_log': None}

    print('--- Standard MoE ---')
    _, mse_log, dyn_log = train_stdmoe(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te)
    all_results['StdMoE'][seed] = {'mse_log': mse_log, 'dyn_log': dyn_log}

    print('--- Nomadic Full ---')
    _, mse_log, dyn_log = train_nomadic(cfg, Xtr, Ytr, Rtr, Xte, Yte, Rte, tags_te)
    all_results['Nomadic'][seed] = {'mse_log': mse_log, 'dyn_log': dyn_log}

    print(f'  Seed {seed} done ({time.time()-t0:.0f}s)')

print('\n=== All seeds complete ===')


In [ ]:
# STEP 10: 결과 집계 + §4.1 기준값 비교
rows = []
for model_name in ['Fixed', 'StdMoE', 'Nomadic']:
    mse_vals, dh_vals, sh_vals, th_vals, lat_vals = [], [], [], [], []
    for seed in SEEDS:
        r = all_results[model_name][seed]
        mse_vals.append(r['mse_log'][-1])
        if r['dyn_log'] is not None:
            d = r['dyn_log'][-1]
            sh_vals.append(d['stable_entropy_mean'])
            th_vals.append(d['transition_entropy_mean'])
            dh_vals.append(d['delta_h'])
            if 'mean_switch_latency' in d:
                lat_vals.append(d['mean_switch_latency'])
    rows.append({
        'Model':        model_name,
        'Variant':      TASK_VARIANT,
        'Seq MSE mean': np.nanmean(mse_vals),
        'Seq MSE std':  np.nanstd(mse_vals),
        'ΔH mean':      np.nanmean(dh_vals)  if dh_vals  else float('nan'),
        'ΔH std':       np.nanstd(dh_vals)   if dh_vals  else float('nan'),
        'Stable Ent':   np.nanmean(sh_vals)  if sh_vals  else float('nan'),
        'Trans Ent':    np.nanmean(th_vals)  if th_vals  else float('nan'),
        'Switch Lat':   np.nanmean(lat_vals) if lat_vals else float('nan'),
    })

df = pd.DataFrame(rows)

print('\n' + '='*70)
print(f'TASK GENERALIZATION: {TASK_VARIANT.upper()} — 3-seed mean')
print('='*70)
print(df.to_string(float_format=lambda x: f'{x:.4f}', index=False))

# §4.1 기준값 비교
print('\n--- vs §4.1 baseline ---')
ref = {'Fixed': (0.409, None), 'StdMoE': (0.410, 0.017), 'Nomadic': (0.165, 0.781)}
for _, row in df.iterrows():
    r_mse, r_dh = ref[row['Model']]
    mse_diff = row['Seq MSE mean'] - r_mse
    dh_diff  = (row['ΔH mean'] - r_dh) if r_dh is not None else float('nan')
    print(f"  {row['Model']:10s} | MSE {row['Seq MSE mean']:.4f} ({mse_diff:+.4f} vs §4.1) "
          f"| ΔH {row['ΔH mean']:.4f} ({dh_diff:+.4f} vs §4.1)")

# 핵심 판정
nomadic_dh  = df[df['Model']=='Nomadic']['ΔH mean'].values[0]
stdmoe_dh   = df[df['Model']=='StdMoE']['ΔH mean'].values[0]
nomadic_mse = df[df['Model']=='Nomadic']['Seq MSE mean'].values[0]
stdmoe_mse  = df[df['Model']=='StdMoE']['Seq MSE mean'].values[0]
print(f'\n[판정]')
print(f'  ΔH gap (Nomadic vs StdMoE): {nomadic_dh - stdmoe_dh:.4f}')
print(f'  MSE gap (Nomadic vs StdMoE): {nomadic_mse - stdmoe_mse:.4f}')
if nomadic_dh > stdmoe_dh + 0.3:
    print('  → entropy differentiation 유지 확인')
else:
    print('  → entropy differentiation 약화 — 추가 분석 필요')


In [ ]:
# STEP 11: 시각화
COLORS = {'Fixed': '#888780', 'StdMoE': '#5DCAA5', 'Nomadic': '#7F77DD'}
LS     = {'Fixed': ':',       'StdMoE': '--',       'Nomadic': '-'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Task Generalization: {TASK_VARIANT.upper()}\n(3-seed mean ± std)',
             fontsize=12, fontweight='bold')

# Left: Seq MSE curves
ax = axes[0]
for mn in ['Fixed', 'StdMoE', 'Nomadic']:
    curves = np.array([all_results[mn][s]['mse_log'] for s in SEEDS])
    m = curves.mean(0); s = curves.std(0)
    ep = np.arange(1, len(m)+1)
    ax.plot(ep, m, color=COLORS[mn], ls=LS[mn], lw=2, label=mn)
    ax.fill_between(ep, m-s, m+s, color=COLORS[mn], alpha=0.12)
# §4.1 기준선
ax.axhline(0.165, color='#7F77DD', lw=0.8, ls=':', alpha=0.5, label='§4.1 Nomadic')
ax.axhline(0.410, color='#5DCAA5', lw=0.8, ls=':', alpha=0.5, label='§4.1 StdMoE')
ax.set_xlabel('Epoch'); ax.set_ylabel('Seq MSE')
ax.set_title('Seq MSE (↓ better)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Middle: ΔH curves
ax2 = axes[1]
for mn in ['StdMoE', 'Nomadic']:
    if all_results[mn][SEEDS[0]]['dyn_log'] is None: continue
    curves = np.array([[d['delta_h'] for d in all_results[mn][s]['dyn_log']] for s in SEEDS])
    m = np.nanmean(curves, 0); s = np.nanstd(curves, 0)
    ep = np.arange(1, len(m)+1)
    ax2.plot(ep, m, color=COLORS[mn], ls=LS[mn], lw=2, label=mn)
    ax2.fill_between(ep, m-s, m+s, color=COLORS[mn], alpha=0.12)
ax2.axhline(0.781, color='#7F77DD', lw=0.8, ls=':', alpha=0.5, label='§4.1 Nomadic')
ax2.axhline(0.017, color='#5DCAA5', lw=0.8, ls=':', alpha=0.5, label='§4.1 StdMoE')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('ΔH')
ax2.set_title('ΔH = Trans H − Stable H (↑ better)'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# Right: Stable / Trans Entropy final bar
ax3 = axes[2]
models_with_dyn = ['StdMoE', 'Nomadic']
x = np.arange(len(models_with_dyn)); w = 0.35
sh_means = [np.nanmean([all_results[mn][s]['dyn_log'][-1]['stable_entropy_mean']     for s in SEEDS]) for mn in models_with_dyn]
th_means = [np.nanmean([all_results[mn][s]['dyn_log'][-1]['transition_entropy_mean'] for s in SEEDS]) for mn in models_with_dyn]
ax3.bar(x-w/2, sh_means, w, label='Stable H',     color=[COLORS[mn] for mn in models_with_dyn], alpha=0.6)
ax3.bar(x+w/2, th_means, w, label='Transition H', color=[COLORS[mn] for mn in models_with_dyn], alpha=1.0)
ax3.set_xticks(x); ax3.set_xticklabels(models_with_dyn)
ax3.set_ylabel('Entropy'); ax3.set_title('Final Entropy (stable vs transition)')
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
fname = f'/content/task_gen_{TASK_VARIANT}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fname}')


In [ ]:
# STEP 12: 결과 저장
import json, os, shutil

SAVE_DIR = f'/content/task_gen_{TASK_VARIANT}'
os.makedirs(SAVE_DIR, exist_ok=True)

df.to_csv(os.path.join(SAVE_DIR, f'results_summary_{TASK_VARIANT}.csv'), index=False)

detail_rows = []
for mn in ['Fixed','StdMoE','Nomadic']:
    for seed in SEEDS:
        r = all_results[mn][seed]
        d = r['dyn_log'][-1] if r['dyn_log'] else {}
        detail_rows.append({
            'Model': mn, 'Variant': TASK_VARIANT, 'Seed': seed,
            'Seq MSE': r['mse_log'][-1],
            'Stable Ent': d.get('stable_entropy_mean', float('nan')),
            'Trans Ent':  d.get('transition_entropy_mean', float('nan')),
            'ΔH':         d.get('delta_h', float('nan')),
            'Switch Lat': d.get('mean_switch_latency', float('nan')),
        })

df_detail = pd.DataFrame(detail_rows)
df_detail.to_csv(os.path.join(SAVE_DIR, f'results_per_seed_{TASK_VARIANT}.csv'), index=False)
print(df_detail.to_string(float_format=lambda x: f'{x:.4f}', index=False))

fig_src = f'/content/task_gen_{TASK_VARIANT}.png'
if os.path.exists(fig_src):
    shutil.copy(fig_src, os.path.join(SAVE_DIR, f'task_gen_{TASK_VARIANT}.png'))

print(f'\nSaved to: {SAVE_DIR}')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  {f}')
